In [1]:
import sys, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
sys.path.insert(0, '../../../scripts')
from score_sentiment import assign_trading_date

In [ ]:

DATA = '../../../outputs/v3/data'
HOLDOUT_START = pd.Timestamp('2026-01-01')
sectors = pd.read_csv(f'{DATA}/universe_2015.csv').set_index('Symbol')['GICS Sector']
in_index = pd.read_csv(f'{DATA}/sp500_membership.csv', index_col=0, parse_dates=True).stack()

### One row per earnings release
When a company files an earnings release, does a change in its AI or capex language predict how its stock does against its sector over the next month?

In [7]:
# 1. Daily returns relative to the sector, index members only
px = pd.read_csv(f'{DATA}/features_df.csv', usecols=['Date', 'Ticker', 'ret_1d', 'ret_20d', 'vol_20d'],
                 index_col=['Date', 'Ticker'], parse_dates=['Date'])
px = px[in_index.reindex(px.index, fill_value=False).values]
sec = px.index.get_level_values('Ticker').map(sectors)
px['excess'] = px['ret_1d'] - px.groupby([px.index.get_level_values('Date'), sec])['ret_1d'].transform('median')
ex = px['excess'].unstack('Ticker').sort_index()          # Date x Ticker

# 2. Window returns: sum of excess returns from day D+a to D+b
cum = ex.fillna(0).cumsum()
def window(a, b):
    return (cum.shift(-b) - cum.shift(-(a - 1))).where(ex.notna()).stack()
HORIZONS = {'drift_5': (2,6), 'drift': (2,21), 'drift_60': (2,61)}
car = pd.DataFrame({'ann':window(0,1), **{k:window(a,b) for k, (a,b) in HORIZONS.items()}})
td = ex.index
# trading date of D+b; beyond the data counts as hold-out
def end_date(b):
    return pd.Series(td, index=td).shift(-b)


# 3. One row per earnings release, with the change since the previous release
f = pd.read_csv(f'{DATA}/filing_signals_df.csv')
f['Date'] = assign_trading_date(f['Accepted'], td)
f = f.dropna(subset=['Date']).sort_values(['Ticker', 'Date']).drop_duplicates(['Ticker', 'Date'], keep='last')
signals = ['ai_demand_net', 'capex_net', 'ai_tone', 'ai_sentence_ratio']
for s in signals:
    f[f'd_{s}'] = f.groupby('Ticker')[s].diff()

ev = f.set_index(['Date', 'Ticker']).join(car, how='inner').join(px[['ret_20d', 'vol_20d']], how='inner')
ev['quarter'] = ev.index.get_level_values('Date').to_period('Q')

# 4. Fama-MacBeth by quarter
controls = ['ann', 'ret_20d', 'vol_20d']
def fama_macbeth(signal, outcome='drift'):
    b = HORIZONS[outcome][1]
    keep = end_date(b).reindex(ev.index.get_level_values('Date')).values < HOLDOUT_START
    slopes = []
    for q, g in ev[keep].dropna(subset=[signal, outcome]).groupby('quarter'):
        r = g[['drift', signal] + controls].rank(pct=True)
        slopes.append(sm.OLS(r['drift'], sm.add_constant(r[[signal] + controls])).fit().params[signal])
    s = pd.Series(slopes)
    m = sm.OLS(s.values, np.ones(len(s))).fit(cov_type='HAC', cov_kwds={'maxlags': 2})
    return s.mean(), m.tvalues[0], m.pvalues[0], len(s), keep.sum()

cols = ['slope', 't', 'p', 'quarters', 'events']
primary = pd.DataFrame([fama_macbeth(f'd_{s}') for s in signals], index=signals, columns=cols)
primary['p_holm'] = multipletests(primary['p'], method='holm')[1]

secondary = pd.DataFrame([(o, s, *fama_macbeth(f'd_{s}', o)) for o in ['drift_5', 'drift_60'] for s in signals],
                         columns=['horizon', 'signal'] + cols).set_index(['horizon', 'signal'])
secondary['p_holm'] = multipletests(secondary['p'], method='holm')[1]

In [8]:
primary

,slope,t,p,quarters,events,p_holm
ai_demand_net,-0.076334,-1.415469,0.156931,28,12760,0.470793
capex_net,-0.035601,-1.922344,0.054562,28,12760,0.218250
ai_tone,0.003992,0.152552,0.878752,28,12760,0.878752
ai_sentence_ratio,-0.017859,-0.890140,0.373391,28,12760,0.746781


In [9]:
secondary

slope         t         p  quarters  events  \
horizon  signal                                                              
drift_5  ai_demand_net     -0.077792 -1.440532  0.149717        28   12791   
         capex_net         -0.035363 -1.917860  0.055129        28   12791   
         ai_tone            0.004748  0.181290  0.856140        28   12791   
         ai_sentence_ratio -0.018295 -0.910393  0.362615        28   12791   
drift_60 ai_demand_net     -0.065325 -1.196584  0.231469        28   12295   
         capex_net         -0.031541 -1.800525  0.071778        28   12295   
         ai_tone            0.002556  0.097631  0.922225        28   12295   
         ai_sentence_ratio -0.018089 -0.900913  0.367634        28   12295   

                              p_holm  
horizon  signal                       
drift_5  ai_demand_net      0.898301  
         capex_net          0.441030  
         ai_tone            1.000000  
         ai_sentence_ratio  1.000000  
drift_60 ai_demand_net      1.000000  
         capex_net          0.502444  
         ai_tone            1.000000  
         ai_sentence_ratio  1.000000